# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library, referencing all dataset entities by their `@id`.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset loaded: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata,'datePublished', None)}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs.

**All entities are referenced by their `@id`**.


In [ ]:
# List all record sets and their fields, referencing by @id
print("Record Sets (@id) and their Fields:")
record_sets = []
for record_set in dataset.metadata.record_sets:
    record_sets.append(record_set['@id'])
    print(f"- RecordSet @id: {record_set['@id']}")
    for field in record_set['fields']:
        print(f"    - Field @id: {field['@id']} \t Label: {field.get('name', '')}")
        # If the field is mapped to a column, show that too
        if 'column' in field:
            column = field['column']
            if isinstance(column, dict):
                print(f"      -> Column @id: {column['@id']}  (name={column.get('name','')})")
            elif isinstance(column, list):
                for c in column:
                    print(f"      -> Column @id: {c['@id']}  (name={c.get('name','')})")
        print()


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s obtained from the overview. All keys below use their full `@id`.


In [ ]:
# Build list of record set @ids programmatically
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Extract records into DataFrames for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet '@id': {record_set_id} - Shape: {df.shape}")
        print(f"Fields (columns/@ids): {df.columns.tolist()[:8]}{' ...' if len(df.columns)>8 else ''}\n")

# Preview the first record set with data
main_record_set_id = next(iter(dataframes))
print(f"Previewing first five records from record set '@id': {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Typical data cleaning and preparation: filtering, normalizing, grouping.

We select a numeric field from the primary record set. All fields referenced by their `@id`.

In [ ]:
# Identify a suitable numeric field @id by inspecting the first record set's DataFrame
numeric_fields = list(dataframes[main_record_set_id].select_dtypes(include=['number']).columns)
print(f"Numeric fields in {main_record_set_id}: {numeric_fields}\n")

# For illustration, select the first numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # This is an @id field (str)
    print(f"Analyzing numeric field: {numeric_field_id}\n")
    
    # Filter records above a threshold (e.g. 10)
    threshold = 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    
    print(f"Filtered rows in {main_record_set_id} where {numeric_field_id} > {threshold}: {len(filtered_df)}")
    print(filtered_df.head(3))
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try to group by a non-numeric field (e.g. first string field)
    group_fields = list(dataframes[main_record_set_id].select_dtypes(include=['object']).columns)
    if group_fields:
        group_field_id = group_fields[0]
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped.head())
else:
    print(f"No numeric fields found in record set {main_record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-whitegrid')

# Visualize distribution for the numeric field, if available
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot if at least two numeric fields
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(
            x=dataframes[main_record_set_id][numeric_fields[0]],
            y=dataframes[main_record_set_id][numeric_fields[1]]
        )
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"Scatter: {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using `mlcroissant`.

**Key Steps:**
- Loaded dataset metadata and records by referencing all Croissant entities by their `@id`.
- Provided an overview of record sets and fields.
- Loaded tabular data for analysis with pandas.
- Applied numeric field filtering, normalization, and grouping operations for EDA.
- Visualized field distributions using Seaborn.

This workflow supports reproducible exploration of ML-ready datasets described with the Croissant schema.
